## 설치

현재 main 기준 설치 예시입니다.

```bash
uv add git+https://github.com/CGINSIDE-ROOKIES/document-processor
# 또는 로컬 체크아웃에서:
uv pip install -e .
```

In [ ]:
from document_processor import DocIR
from pprint import pprint

pprint(DocIR.model_fields)

- `identity_version`: 현 미사용 (versioning이나 스키마 버전용)
- `meta`: 필요한 메타데이터 있을 경우 여기다 원하는 데이터 추가 가능 (DataFrame 상속 obj)

In [ ]:
from pathlib import Path

DOC_DIR = Path("doc_samples/new_test/")

# 예시
FP = DOC_DIR / "01. 대중문화예술분야 연습생 표준계약서.hwpx"

doc = DocIR.from_file(FP)

In [ ]:
doc.source_doc_type, doc.source_path, doc.doc_id

각 페이지 정보

In [ ]:
[pprint(page) for page in doc.pages]
print(len(doc.pages))

In [ ]:
len(doc.paragraphs)

In [ ]:
[(para.node_id, para.text) for para in doc.paragraphs][:10]

In [ ]:
sample = doc.paragraphs[0]

print(sample.model_dump_json(indent=2))

In [ ]:
sample.runs, sample.images

In [ ]:
sample.para_style

표(table) 다루기

In [ ]:
tables = [table for para in doc.paragraphs for table in para.tables]
len(tables), tables

In [ ]:
table = tables[0]
(table.row_count, table.col_count), \
    [(cell.row_index, cell.col_index, cell.node_id, cell.text) for cell in table.cells]

---

## HTML로 출력하기

In [ ]:
html = doc.to_html(title="demo")

In [ ]:
from IPython.core.display import HTML

def show_html(html: str) -> None:
	display(HTML(f"""
		<iframe srcdoc='{html}' width="100%" height="750px" style="border:none;"></iframe>
	"""))

show_html(html)

In [ ]:
from random import randrange
from document_processor import render_review_html, DocumentInput, TextAnnotation

para_list = [para for para in doc.paragraphs]
run_list = [run for para in para_list for run in para.runs]

def get_rand_elem(ls: list):
    res = ls[randrange(len(ls))]
    while not res.text.strip():
        res = ls[randrange(len(ls))]
    return res

rand1 = get_rand_elem(run_list)
rand2 = get_rand_elem(run_list)
rand3 = get_rand_elem(para_list)

doc_input = DocumentInput(doc_ir=doc)

In [ ]:
annotated_result = render_review_html(
	document=doc_input,
	annotations=[
		TextAnnotation(
			target_kind="run",  # AnnotationTargetKind = Literal["paragraph", "run"]
            target_id=rand1.node_id,
            selected_text=rand1.text,
            occurrence_index=0,
            label="match",
            color="#727cff"
		),
		TextAnnotation(
			target_kind="run",
			target_id=rand2.node_id,
			selected_text=rand2.text,
			occurrence_index=0,
			label="match",
			color="#ff6e6e"
		),
		TextAnnotation(
			target_kind="paragraph",
			target_id=rand3.node_id,
			selected_text=rand3.text,
			occurrence_index=0,
			label="하늘샛 하이라이팅!",
			color="#6effd8"
		)
	]
)

f'result validation ok?: {annotated_result.ok}', \
    (rand1.node_id, rand1.text), (rand2.node_id, rand2.text), (rand3.node_id, rand3.text)

In [ ]:
show_html(annotated_result.html)

## 문서 수정하기

In [ ]:
from document_processor import apply_document_edits, TextEdit

# cell만 다루기 - 예시용 셀 
rand4 = [ cell 
	for para in doc.paragraphs 
		for table in para.tables
			for cell in table.cells
][1]

assert rand4.node_id  # 린터 경고 없에기

edited_result = apply_document_edits(
	document=doc_input,
	edits=[
		TextEdit(
            target_kind="run",  # Literal["paragraph", "run", "cell"]
            target_id=rand1.node_id,
            expected_text=rand1.text,
            new_text="Hello Legal World",
            reason="Expand wording",
        ),
		TextEdit(
            target_kind="paragraph",
            target_id=rand3.node_id,
            expected_text=rand3.text,
            new_text="Hello Legal World",
            reason="Expand wording",
        ),
		TextEdit(
            target_kind="cell",
            target_id=rand4.node_id,
            expected_text=rand4.text,
            new_text="Hello Legal World",
            reason="Expand wording",
        ),
	],
 	return_doc_ir=True
)

f'result validation ok?: {edited_result.ok}', \
    (rand1.node_id, rand1.text), (rand3.node_id, rand3.text), (rand4.node_id, rand4.text)

In [ ]:
from document_processor import ApplyDocumentEditsResult
ApplyDocumentEditsResult.model_fields

위처럼 `edited_result.ok`가 False인 경우 exception 처리가 안나오게 설계됨, `.validation.issues`통해서 오류 확인 가능 (langrgaph/langchain/pydanticAI 처리위함)

In [ ]:
from document_processor import EditValidationIssue
EditValidationIssue.model_fields

In [ ]:
edited_result.validation.issues

오류문구:
```
Cell text replacement for cell_c140d37cf7f53629 must preserve paragraph count: expected 2 line(s), got 1.
```
`\n` 통해 paragraph break인식 시키도록 하면 됨

In [ ]:
edited_result = apply_document_edits(
	document=doc_input,
	edits=[
		TextEdit(
            target_kind="run",  # Literal["paragraph", "run", "cell"]
            target_id=rand1.node_id,
            expected_text=rand1.text,
            new_text=" Hello Legal World ",
            reason="Expand wording",
        ),
		TextEdit(
            target_kind="paragraph",
            target_id=rand3.node_id,
            expected_text=rand3.text,
            new_text="\n[Paragraph replaced here!]\n",
            reason="Expand wording",
        ),
		TextEdit(
            target_kind="cell",
            target_id=rand4.node_id,
            expected_text=rand4.text,
            new_text="Hello!\n Legal World",  # \n 통해서 paragraph 분리
            reason="Expand wording",
        ),
	],
 	return_doc_ir=True
)

f'result validation ok?: {edited_result.ok}', \
    (rand1.node_id, rand1.text), (rand3.node_id, rand3.text), (rand4.node_id, rand4.text)

In [ ]:
show_html(edited_result.updated_doc_ir.to_html())

stateless API이므로 수정된거로 input 객체 변경

In [ ]:
doc_input  = DocumentInput(doc_ir=edited_result.updated_doc_ir)

In [ ]:
from document_processor import StructuralEdit
assert rand4.node_id  # 린터 경고 없에기

'''
operation 종류들
Literal[
    "insert_paragraph",
    "remove_paragraph",
    "insert_run",
    "remove_run",
    "insert_table",
    "remove_table",
    "set_cell_text",
    "insert_table_row",
    "remove_table_row",
    "insert_table_column",
    "remove_table_column",
]
'''

struct_edit_result = apply_document_edits(
	document=doc_input,
	edits=[
		StructuralEdit(
			operation="insert_paragraph",
			target_id=rand3.node_id,
			position="before",
			text="paragraph 추가된거임!!!!!"
		),
		StructuralEdit(
			operation="insert_paragraph",
			target_id=rand3.node_id,
			position="after",
			text="paragraph 추가된거임!!!!!"
		),
		StructuralEdit(
			operation="insert_table_row",
			target_id=rand4.node_id,  # cell id 기준으로 행 작업 수행함
			position="after",
			values=['행 추가된겨']
		),
		StructuralEdit(
			operation="insert_table_column",
			target_id=rand4.node_id,  # cell id 기준으로 렬 작업 수행함
			position="after",
			values=['1','2','3','4',]
		),
	],
 	return_doc_ir=True
)

f'result validation ok?: {struct_edit_result.ok}', \
    (rand3.node_id, rand1.text), (rand3.node_id, rand3.text)

In [ ]:
struct_edit_result.created_target_ids

**연달아 추가된 작업의 경우에는 순차적으로 진행**

In [ ]:
show_html(struct_edit_result.updated_doc_ir.to_html())

## 복합 작업 + 스타일 수정

In [ ]:
from document_processor import StyleEdit
assert rand4.node_id  # 린터 경고 없에기

total_edit_result = apply_document_edits(
	document=doc_input,
	edits=[
		StructuralEdit(
			operation="insert_paragraph",
			target_id=rand3.node_id,
			position="before",
			text="paragraph 추가된거임!!!!!"
		),
		StructuralEdit(
			operation="insert_paragraph",
			target_id=rand3.node_id,
			position="after",
			text="paragraph 추가된거임!!!!!"
		),
		StructuralEdit(
			operation="insert_table_row",
			target_id=rand4.node_id,  # cell id 기준으로 행 작업 수행함
			position="after",
			values=['행 추가된겨']
		),
		TextEdit(
            target_kind="paragraph",
            target_id=rand3.node_id,
            expected_text="\n[Paragraph replaced here!]\n",
            new_text="이제 여기 색갈도 바뀜",
            reason="Expand wording",
        ),
		StructuralEdit(
			operation="insert_table_column",
			target_id=rand4.node_id,  # cell id 기준으로 열 작업 수행함
			position="after",
			values=['1','2','3','4',]
		),
		StyleEdit(
            target_kind="run",  # Literal["paragraph", "run", "cell", "table", "image"]
            target_id=rand3.runs[0].node_id,
            bold=True,
            color="#4C3EB8",
            font_size_pt=32,
        ),
		*[  # list unpacking 통해서 안에 여러개 동시 주입 가능!!
			StyleEdit(
				target_kind="run",
				target_id=run.node_id,
				color="#FFFFFF",
			)
			for para in rand4.paragraphs for run in para.runs
       	],
		*[
			StyleEdit(
				target_kind="paragraph",
				target_id=para.node_id,
				paragraph_align="right"  # 여기 주목! ↓
			)
			for para in rand4.paragraphs
       	],
		StyleEdit(
            target_kind="cell",
            target_id=rand4.node_id,
            background="#600B4F",
            padding_left_pt=10,
            padding_right_pt=10,
            height_pt=100,
            width_pt=200,
            horizontal_align="left",    # 여기 주목! ↑
            border_top="5pt single #000000",
			border_right="5pt single #000000",
			border_bottom="5pt single #000000",
			border_left="5pt single #000000",
        ),
	],
	return_doc_ir=True,
)

"#000000"

f'result validation ok?: {total_edit_result.ok}', \
    rand4.node_id

렌더링 우선순위 때문에 cell "horizontal_align" vs paragraph "paragraph_align" 둘이 뜨면 paragraph 렌더링이 우선순위를 가지게 됨, 다음 예시에서도 paragraph상에서 오른정렬한게 셀 단위 왼정렬한걸 이김

In [ ]:
show_html(total_edit_result.updated_doc_ir.to_html())